

**Text embedded per row:** `points_raised` + `conclusion` concatenated into one string.  
Extra columns in the CSV are ignored.

**Output files:**
- `embeddings/embeddings.npy` — matrix of shape (n_rows × 1024)
- `embeddings/records.csv` — original CSV with `embedding_idx` column added

**To add legislation later:** embed chunks with `passage:` prefix and compare against this matrix.

In [ ]:

# !pip install sentence-transformers numpy pandas tqdm

In [1]:
import numpy as np
import pandas as pd
from pathlib import Path
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from numpy.linalg import norm

/Users/inge/Desktop/ITU/Thesis/Network/eu-network-graph/venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


## 1. Load data

In [ ]:
DATA_PATH = "data/commission_meetings_rows.csv"  # ← update to your filename

df = pd.read_csv(DATA_PATH)

print(f"Rows:    {len(df)}")
print(f"Columns: {list(df.columns)}")

assert "points_raised" in df.columns, "Missing column: points_raised"
assert "conclusions"    in df.columns, "Missing column: conclusion"

print(f"\npoints_raised non-null: {df['points_raised'].notna().sum()} / {len(df)}")
print(f"conclusion    non-null: {df['conclusions'].notna().sum()} / {len(df)}")

# Filter to one commissioner
COMMISSIONER = "Valdis Dombrovskis"  # ← change as needed
df = df[df["commissioner_name"] == COMMISSIONER].reset_index(drop=True)
print(f"Rows after filter: {len(df)}")

Rows:    11114
Columns: ['id', 'commissioner_name', 'commissioner_portfolio', 'host_id', 'meeting_type', 'meeting_date', 'location', 'subject', 'commission_representatives', 'organizations_raw', 'transparency_register_ids', 'points_raised', 'conclusions', 'ares_number', 'minutes_url', 'matched_procedure_id', 'match_confidence', 'match_method', 'source_url', 'raw_data', 'created_at', 'updated_at', 'actor_id']

points_raised non-null: 9788 / 11114
conclusion    non-null: 4699 / 11114
Rows after filter: 589


## 2. Build text to embed

Concatenate `points_raised` and `conclusion` into one string per row.
Rows where both are empty/N/A get an empty string and will receive a zero vector.

In [16]:
SKIP_VALUES = {"n/a", "na", "none", "nan", ""}  # values treated as empty

def build_text(row):
    parts = []
    for col in ["points_raised", "conclusions"]:
        val = str(row[col]).strip() if pd.notna(row[col]) else ""
        if val.lower() not in SKIP_VALUES:
            parts.append(val)
    return " ".join(parts)

df["_text"] = df.apply(build_text, axis=1)

n_empty = (df["_text"].str.strip() == "").sum()
print(f"Rows with text to embed:   {len(df) - n_empty}")
print(f"Rows with no text (empty): {n_empty}  ← will get zero vector")
print(f"\nExample concatenated text (row 0):")
print(df["_text"].iloc[0])

Rows with text to embed:   563
Rows with no text (empty): 26  ← will get zero vector

Example concatenated text (row 0):
The main point of the meeting was the economic security strategy by Commission, especially for technologies and reducing high-risk dependencies. At the meeting they also discussed the simplification efforts, including digital omnibus to scale up innovations. Google explains investment in the EU and approach on AI developments.


## 3. Load model

`multilingual-e5-large` — 560M params, 1024-dim embeddings.  
Downloads ~2.2 GB on first run, cached locally afterwards.  
Swap to `"intfloat/multilingual-e5-base"` for a faster/smaller version.

In [8]:
MODEL_NAME = "intfloat/multilingual-e5-large"

print(f"Loading {MODEL_NAME} ...")
model = SentenceTransformer(MODEL_NAME)
print(f"Done. Embedding dimension: {model.get_sentence_embedding_dimension()}")

Loading intfloat/multilingual-e5-large ...


modules.json:   0%|          | 0.00/387 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/690 [00:00<?, ?B/s]

...

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/280 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/201 [00:00<?, ?B/s]

Done. Embedding dimension: 1024


## 4. Embed

E5 uses a prefix to signal text type:
- `query:` → meeting notes  
- `passage:` → documents — use this later when embedding EUR-Lex legislation chunks

In [17]:
BATCH_SIZE = 32  # reduce to 16 if you run out of memory
DIM = model.get_sentence_embedding_dimension()

# Only encode rows that have content
encode_mask    = df["_text"].str.strip() != ""
encode_indices = df.index[encode_mask].tolist()
texts_to_encode = [f"query: {df['_text'].iloc[i]}" for i in encode_indices]

print(f"Encoding {len(texts_to_encode)} rows in batches of {BATCH_SIZE}...")
print(f"Estimated time: ~{len(texts_to_encode) // BATCH_SIZE // 2 + 1} min on CPU\n")

vectors = model.encode(
    texts_to_encode,
    batch_size=BATCH_SIZE,
    show_progress_bar=True,
    normalize_embeddings=True,  # L2-normalise so cosine similarity == dot product
    convert_to_numpy=True,
)

# Build full matrix — zero vector for empty rows
embeddings = np.zeros((len(df), DIM), dtype=np.float32)
for pos, row_idx in enumerate(encode_indices):
    embeddings[row_idx] = vectors[pos]

print(f"\nEmbedding matrix shape: {embeddings.shape}")

Encoding 563 rows in batches of 32...
Estimated time: ~9 min on CPU



Batches:   0%|          | 0/18 [00:00<?, ?it/s]


Embedding matrix shape: (589, 1024)


## 5. Save

In [18]:
OUTPUT_DIR = Path("embeddings")
OUTPUT_DIR.mkdir(exist_ok=True)

# Embedding matrix
np.save(OUTPUT_DIR / "validis.npy", embeddings)
print(f"Saved embeddings.npy  shape={embeddings.shape}  ({embeddings.nbytes / 1e6:.1f} MB)")

# CSV with embedding_idx added (== row number, since data is already deduplicated)
df_out = df.drop(columns=["_text"]).copy()
df_out["embedding_idx"] = range(len(df_out))
df_out.to_csv(OUTPUT_DIR / "validis_records.csv", index=False)
print(f"Saved validis_records.csv  ({len(df_out)} rows)")

Saved embeddings.npy  shape=(589, 1024)  (2.4 MB)
Saved validis_records.csv  (589 rows)


## 6. Sanity checks

In [31]:
# ── Check 1: similar arguments should score high, unrelated ones low ──────────
print("── Similarity sanity check ──\n")

test_pairs = [
    (
        "The organisation urged the Commission to reconsider CBAM implementation timelines.",
        "Representatives stressed that the carbon border adjustment schedule creates disproportionate burden for SMEs.",
        "SHOULD BE HIGH — same argument, different words"
    ),
    (
        "Representatives from L’Oreal Poland outlined their position on the Urban Wasterwater Treatment Directive. They reported on the assessment carried out by their companies re implementation challenges linked to this Directive. Cab Dombrovskis took note of L’Oreal’s position.",
        "The delegation presented their views on fisheries quota allocations in the North Sea.",
        "SHOULD BE LOW — unrelated topics"
    ),
]

for text_a, text_b, label in test_pairs:
    vec_a = model.encode(f"query: {text_a}", normalize_embeddings=True)
    vec_b = model.encode(f"query: {text_b}", normalize_embeddings=True)
    score = float(np.dot(vec_a, vec_b))
    print(f"  {label}")
    print(f"  Score: {score:.3f}\n")

── Similarity sanity check ──

  SHOULD BE HIGH — same argument, different words
  Score: 0.846

  SHOULD BE LOW — unrelated topics
  Score: 0.781



In [30]:
# ── Check 2: nearest neighbours for a sample row ──────────────────────────────
print("── Nearest neighbours for a sample row ──\n")

# Pick first row with a decent-length text
sample_idx = 20
sample_row = df_out.iloc[sample_idx]

print(f"Query (row {sample_idx}):")
print(f"  points_raised: {str(sample_row.get('points_raised', ''))[:400]}")
print(f"  conclusion:    {str(sample_row.get('conclusion', ''))[:150]}\n")

scores = embeddings @ embeddings[sample_idx]
scores[sample_idx] = -1  # exclude self
top5 = np.argsort(scores)[::-1][:5]

print("Top 5 most similar rows:")
for rank, idx in enumerate(top5, 1):
    row = df_out.iloc[idx]
    print(f"  #{rank}  score={scores[idx]:.3f}  row={idx}")
    print(f"       points_raised: {str(row.get('points_raised', ''))[:550]}")
    print()

── Nearest neighbours for a sample row ──

Query (row 20):
  points_raised: - The representatives shared their view on the responsible European Parliament Committee’s report related to the Omnibus IV.
- They also reported on implementation challenges linked to the Packaging and Packaging Waste Regulation, where they stressed the need for clear guidance from the Commission.
- Commission took note of the Cosmetics Europe’s positions and explained that guidance on PPWR will 
  conclusion:    

Top 5 most similar rows:
  #1  score=0.903  row=237
       points_raised: Representatives from the FPE shared their views on the environmental omnibus, including the Packaging Waste Regulation and Extended Producer Responsibility. They stressed their opposition to re-open the Regulation and argued in favour of well-defined guidance, providing much needed clarity on a number of key aspects of the Regulation. The Commission’s representative took note of their views and stressed that guidance on the PP

/var/folders/99/scjg2kvj5g3477xxgx_4_ghw0000gn/T/ipykernel_78303/3052849959.py:12: RuntimeWarning: divide by zero encountered in matmul
  scores = embeddings @ embeddings[sample_idx]
/var/folders/99/scjg2kvj5g3477xxgx_4_ghw0000gn/T/ipykernel_78303/3052849959.py:12: RuntimeWarning: overflow encountered in matmul
  scores = embeddings @ embeddings[sample_idx]
/var/folders/99/scjg2kvj5g3477xxgx_4_ghw0000gn/T/ipykernel_78303/3052849959.py:12: RuntimeWarning: invalid value encountered in matmul
  scores = embeddings @ embeddings[sample_idx]


## 7. How to load and use in future notebooks

Copy this block — you never need to re-embed unless you change the model.

In [ ]:
# import numpy as np
# import pandas as pd
# from sentence_transformers import SentenceTransformer
#
# embeddings = np.load("embeddings/embeddings.npy")
# df         = pd.read_csv("embeddings/records.csv")
#
# # Similarity between row i and all others:
# scores = embeddings @ embeddings[i]
#
# # Add legislation later — no re-embedding of meetings needed:
# model   = SentenceTransformer("intfloat/multilingual-e5-large")
# law_vec = model.encode("passage: Article 4 shall apply to...", normalize_embeddings=True)
# scores  = embeddings @ law_vec
# top5    = np.argsort(scores)[::-1][:5]

print("See comments above.")

## 7b. Parse and embed legislation PDF

Chunks the legislation into recitals and Article 1 paragraphs,
then embeds each chunk with a `passage:` prefix.

Output:
- `legislation_embeddings/legislation_chunks.json`
- `legislation_embeddings/legislation_embeddings.npy`

In [41]:
import re
import json
import numpy as np
import pdfplumber
from pathlib import Path

PDF_PATH = "COM_COM_2025_0836_EN.pdf"  # ← update to your filename

# ── 1. Extract full text ─────────────────────────────────────────────────────
def extract_text(pdf_path):
    text = ""
    with pdfplumber.open(pdf_path) as pdf:
        for page in pdf.pages:
            t = page.extract_text()
            if t:
                text += t + "\n"
    return text

# ── 2. Find section boundaries ───────────────────────────────────────────────
def find_bounds(text):
    bounds = {}
    for key, pattern in [
        ("recitals_start", r'Whereas:\n'),
        ("recitals_end",   r'HAVE ADOPTED THIS REGULATION'),
        ("article1_start", r'\nArticle 1\nAmendments'),
        ("article2_start", r'\nArticle 2\n'),
    ]:
        m = re.search(pattern, text)
        bounds[key] = m.start() if m else None
    return bounds

# ── 3. Parse recitals ────────────────────────────────────────────────────────
def parse_recitals(text, start, end):
    section = text[start:end]
    chunks, matches = [], list(re.compile(r'\((\d+)\)\s+').finditer(section))
    for i, match in enumerate(matches):
        num = int(match.group(1))
        content = section[match.end(): matches[i+1].start() if i+1 < len(matches) else len(section)]
        content = re.sub(r'\s+', ' ', content).strip()
        if len(content) < 50:
            continue
        chunks.append({"chunk_id": f"recital_{num}", "type": "recital",
                        "number": num, "text": content})
    return chunks

# ── 4. Parse Article 1 paragraphs ────────────────────────────────────────────
def parse_article1(text, start, end):
    section = text[start:end]
    chunks, matches = [], list(re.compile(r'\n\((\d+)\)\s+').finditer(section))
    for i, match in enumerate(matches):
        num = int(match.group(1))
        content = section[match.end(): matches[i+1].start() if i+1 < len(matches) else len(section)]
        content = re.sub(r'\s+', ' ', content).strip()
        if len(content) < 30:
            continue
        ref = re.search(r'(?:in |Article |amend(?:ing|s)?\s+)Article\s+([\d\w\(\)]+)',
                         content[:150], re.IGNORECASE)
        chunks.append({"chunk_id": f"article1_para_{num}", "type": "article1_paragraph",
                        "number": num, "ai_act_ref": ref.group(0).strip() if ref else "",
                        "text": content})
    return chunks

# ── 5. Run chunking ──────────────────────────────────────────────────────────
print(f"Extracting text from {PDF_PATH} ...")
leg_text   = extract_text(PDF_PATH)
bounds     = find_bounds(leg_text)
recitals   = parse_recitals(leg_text, bounds['recitals_start'], bounds['recitals_end'])
articles   = parse_article1(leg_text, bounds['article1_start'], bounds['article2_start'])
leg_chunks = recitals + articles

for i, c in enumerate(leg_chunks):
    c['embedding_idx'] = i

print(f"  Recitals:            {len(recitals)}")
print(f"  Article 1 paragraphs: {len(articles)}")
print(f"  Total chunks:         {len(leg_chunks)}")


Extracting text from COM_COM_2025_0836_EN.pdf ...
  Recitals:            46
  Article 1 paragraphs: 33
  Total chunks:         79


In [42]:
# ── 6. Embed with passage: prefix ────────────────────────────────────────────
# Reuses the model already loaded above in cell 3

texts = [f"passage: {c['text']}" for c in leg_chunks]

print(f"Embedding {len(texts)} legislation chunks...")
leg_embeddings = model.encode(
    texts,
    batch_size=16,
    show_progress_bar=True,
    normalize_embeddings=True,
    convert_to_numpy=True,
)
print(f"Shape: {leg_embeddings.shape}")

# ── 7. Save ───────────────────────────────────────────────────────────────────
LEG_DIR = Path("legislation_embeddings")
LEG_DIR.mkdir(exist_ok=True)

np.save(LEG_DIR / "legislation_embeddings.npy", leg_embeddings)
with open(LEG_DIR / "legislation_chunks.json", "w", encoding="utf-8") as f:
    json.dump(leg_chunks, f, ensure_ascii=False, indent=2)

print(f"Saved legislation_embeddings.npy  shape={leg_embeddings.shape}")
print(f"Saved legislation_chunks.json     ({len(leg_chunks)} chunks)")

# Quick preview
print("\nFirst 5 chunk IDs:")
for c in leg_chunks[:5]:
    print(f"  {c['chunk_id']:<25} {c['text'][:80]}...")


Embedding 79 legislation chunks...


Batches:   0%|          | 0/5 [00:00<?, ?it/s]

Shape: (79, 1024)
Saved legislation_embeddings.npy  shape=(79, 1024)
Saved legislation_chunks.json     (79 chunks)

First 5 chunk IDs:
  recital_1                 Regulation (EU) 2024/1689 of the European Parliament and of the Council3 lays do...
  recital_2                 The experience gathered in implementing the parts of Regulation (EU) 2024/1689 t...
  recital_3                 Consequently, targeted amendments to Regulation (EU) 2024/1689 are necessary to ...
  recital_4                 Enterprises outgrowing the micro, small and medium-sized enterprises (‘SME’) def...
  recital_5                 Article 4 of Regulation (EU) 2024/1689 currently imposes an obligation on all pr...


In [ ]:
import numpy as np
import pandas as pd
import json

# Load everything
meet_emb = np.load("embeddings/embeddings.npy")
meet_df  = pd.read_csv("embeddings/records.csv")



with open("legislation_embeddings/legislation_chunks.json") as f:
    leg_chunks = json.load(f)
leg_emb = np.load("legislation_embeddings/legislation_embeddings.npy")
print("meeting emb norm:", np.linalg.norm(meet_emb[0]))
print("legislation emb norm:", np.linalg.norm(leg_emb[0]))
# Build a quick lookup: chunk_id → index
chunk_lookup = {c["chunk_id"]: i for i, c in enumerate(leg_chunks)}

# Query function — given a chunk ID, show the chunk and top 5 most similar meeting rows
def top5(chunk_id):
    idx = chunk_lookup[chunk_id]
    vec = leg_emb[idx]
    
    chunk = leg_chunks[idx]
    print(f"CHUNK: {chunk_id}")
    print(f"TYPE:  {chunk['type']}")
    if chunk.get("ai_act_ref"):
        print(f"AMENDS: {chunk['ai_act_ref']}")
    print(f"TEXT:  {chunk['text'][:800]}")
    print()
    
    scores = meet_emb @ vec
    top    = np.argsort(scores)[::-1][:5]
    
    for rank, i in enumerate(top, 1):
        row = meet_df.iloc[i]
        print(f"  #{rank}  score={scores[i]:.3f}")
        print(f"  org:  {str(row.get('interest_representatives', ''))[:70]}")
        print(f"  date: {row.get('date', '')}")
        print(f"  text: {str(row.get('points_raised', ''))[:800]}")
        print()

# ── List all available chunk IDs ──────────────────────────────────────────────
def list_chunks():
    for c in leg_chunks:
        label = c.get("ai_act_ref") or c.get("section") or ""
        print(f"  {c['chunk_id']:<30}  {label[:60]}")

# ── Usage ─────────────────────────────────────────────────────────────────────
list_chunks()

meeting emb norm: 1.0
legislation emb norm: 1.0
  recital_1                       
  recital_2                       
  recital_3                       
  recital_4                       
  recital_5                       
  recital_6                       
  recital_7                       
  recital_8                       
  recital_3                       
  recital_9                       
  recital_3                       
  recital_2                       
  recital_3                       
  recital_10                      
  recital_1                       
  recital_11                      
  recital_12                      
  recital_13                      
  recital_14                      
  recital_1                       
  recital_9                       
  recital_15                      
  recital_16                      
  recital_1                       
  recital_17                      
  recital_18                      
  recital_19                      
  recit

In [39]:

top5("recital_6")       # bias detection / personal data


CHUNK: recital_6
TYPE:  recital
TEXT:  Bias detection and correction constitute a substantial public interest because they protect natural persons from biases’ adverse effects, including discrimination. Discrimination might result from the bias in AI models and AI systems other than high-risk AI systems for which of Regulation (EU) 2024/1689 already provides a legal basis authorising the processing of special categories of personal data under Article 9(2), point (g), of Regulation (EU) 2016/679 of the European Parliament and of the Council6. Given that discrimination might result also from those other AI systems and models, it is therefore appropriate that Regulation (EU) 2024/1689 should provide for a legal basis for the processing of special categories of personal data also by providers and deployers of other AI systems and A

  #1  score=0.823
  org:  
  date: 
  text: - The interest representatives presented their views on the proposed EU regulation on SEPs. They voiced their suppo

CHUNK: recital_6
TYPE:  recital
TEXT:  Bias detection and correction constitute a substantial public interest because they protect natural persons from biases’ adverse effects, including discrimination. Discrimination might result from the bias in AI models and AI systems other than high-risk AI systems for which of Regulation (EU) 2024/1689 already provides a legal basis authorising the processing of special categories of personal data under Article 9(2), point (g), of Regulation (EU) 2016/679 of the European Parliament and of the Council6. Given that discrimination might result also from those other AI systems and models, it is therefore appropriate that Regulation (EU) 2024/1689 should provide for a legal basis for the processing of special categories of personal data also by providers and deployers of other AI systems and A

  #1  score=0.823
  org:  
  date: 
  text: - The interest representatives presented their views on the proposed EU regulation on SEPs. They voiced their suppo

/var/folders/99/scjg2kvj5g3477xxgx_4_ghw0000gn/T/ipykernel_78303/188644291.py:29: RuntimeWarning: divide by zero encountered in matmul
  scores = meet_emb @ vec
/var/folders/99/scjg2kvj5g3477xxgx_4_ghw0000gn/T/ipykernel_78303/188644291.py:29: RuntimeWarning: overflow encountered in matmul
  scores = meet_emb @ vec
/var/folders/99/scjg2kvj5g3477xxgx_4_ghw0000gn/T/ipykernel_78303/188644291.py:29: RuntimeWarning: invalid value encountered in matmul
  scores = meet_emb @ vec
